# 03 — Class Comparison

Compare two combatant builds side-by-side: a high-STR Warrior vs
a high-DEX Warrior (same weapon, same target).

In [ ]:
from pathlib import Path

from omega.model.constants import SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_ANATOMY
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import comparison_table, format_table_html
from omega.reporting.plots import comparison_overlay

SHARD_ROOT = Path("..") / "submodules" / "zuluhotel_omega_2.5"
shard = ShardData.from_path(SHARD_ROOT)
parse_results = shard.parse_combat_scripts()

In [ ]:
SHARED_WEAPON = WeaponSpec(name="Broadsword", damage="3d6+2")
SHARED_DEFENDER = CombatantSpec(
    name="Target", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    armor=ArmorSpec(ar=30),
)
SHARED_SKILLS = {SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100}
RUN_KW = dict(
    parse_results=parse_results,
    config_resolver=shard.resolve_config_path,
    em_modules_dir=shard.root / "scripts" / "modules",
)

# Build A: STR Warrior (str 120, dex 80)
str_result = run_scenario(
    Scenario(
        attacker=CombatantSpec(
            name="STR Warrior",
            skills=SHARED_SKILLS, str_=120, dex_=80, int_=25,
            class_levels={"IsWarrior": 5},
            weapon=SHARED_WEAPON,
        ),
        defender=SHARED_DEFENDER,
        iterations=200, base_seed=42,
    ),
    **RUN_KW,
)

# Build B: DEX Warrior (str 80, dex 120)
dex_result = run_scenario(
    Scenario(
        attacker=CombatantSpec(
            name="DEX Warrior",
            skills=SHARED_SKILLS, str_=80, dex_=120, int_=25,
            class_levels={"IsWarrior": 5},
            weapon=SHARED_WEAPON,
        ),
        defender=SHARED_DEFENDER,
        iterations=200, base_seed=42,
    ),
    **RUN_KW,
)

print(f"STR: {str_result.damage_stats.mean:.2f} mean, DEX: {dex_result.damage_stats.mean:.2f} mean")

In [ ]:
# Overlaid histograms
comparison_overlay(
    {"STR Warrior": str_result, "DEX Warrior": dex_result},
    title="STR vs DEX Warrior — Damage Distribution",
)

In [ ]:
# Side-by-side comparison table
from IPython.display import HTML

rows = comparison_table(
    {"STR Warrior": str_result, "DEX Warrior": dex_result},
    stats=["mean", "median", "min", "max", "p5", "p95", "hit_rate"],
)
HTML(format_table_html(rows))